# Buscador agro · cargar una imagen y buscar

Búsqueda configurable de **flores** y **plantas de maleza** en **JPG, PNG, TIFF/GeoTIFF,
BMP, WebP y GIF**, o en imágenes incluidas dentro de **ZIP y 7z**.
Pulsá **Cargar imagen / ZIP / 7z**, elegí un archivo de tu computadora y pulsá
**Buscar en la imagen**. Si el comprimido contiene varias imágenes, elegí una en la lista.
No hay que escribir una ruta ni conectar Drive.

Muestra una imagen con las detecciones y descarga CSV, JSON, recortes y ZIP.
Si el GeoTIFF tiene georreferenciación, agrega puntos y cajas GeoJSON para QGIS.
Si es una foto común o un TIFF sin CRS, trabaja en píxeles, sin inventar coordenadas.
La referencia de 1–2 cm/píxel sólo aplica al ortomosaico cuando su información permite calcularla.

La búsqueda por texto es exploratoria; aún no hay validación con un lote real.
Una planta detectada no confirma especie ni condición de maleza.
El modelo puede fallar por sombras, solapamiento, etapa del cultivo o tamaño del objeto.


## 1. Dependencias

In [ ]:
# Ejecutar una vez. Reiniciar kernel si se actualizan librerias ya cargadas.
%pip install -q ultralytics==8.3.228 https://github.com/ultralytics/CLIP/archive/a13192f8cb767260d7dfd98c843b0716593169e7.zip torch==2.8.0 torchvision==0.23.0 numpy==2.2.6 opencv-python==4.12.0.88 Pillow==11.3.0 pandas==2.3.3 reportlab==4.5.0 openpyxl==3.1.5 ipywidgets==8.1.8 rasterio==1.4.4 pyproj==3.7.2 nbformat==5.10.4 py7zr==1.0.0 ipykernel==6.30.1

## 2. Motor (incluido; no necesita archivos externos)

In [ ]:
"""Search flowers/plants in uploaded photos and GeoTIFFs; coordinates when available."""
from __future__ import annotations

import argparse
from collections import Counter, defaultdict
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import csv
import importlib.metadata
import json
import logging
import math
import os
import re
from pathlib import Path, PurePosixPath
import shutil
import stat
import time
import uuid
import warnings
from typing import Callable, Iterator
import zipfile

import cv2
import numpy as np
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from PIL import Image, ImageOps
from pyproj import CRS, Transformer

LOGGER = logging.getLogger(__name__)
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp", ".gif"}
UPLOAD_EXTENSIONS = IMAGE_EXTENSIONS | {".zip", ".7z"}
MAX_ARCHIVE_BYTES = 20 * 1024**3
MAX_ARCHIVE_MEMBERS = 10_000


def file_path(value: str | Path) -> Path:
    """Use Windows extended paths so deep OneDrive folders remain writable."""
    path = str(Path(value).absolute())
    if os.name == "nt" and not path.startswith("\\\\?\\"):
        path = "\\\\?\\UNC\\" + path[2:] if path.startswith("\\\\") else "\\\\?\\" + path
    return Path(path)


def save_uploaded_image(name: str, content: bytes | memoryview, folder: str | Path) -> Path:
    """Save one browser upload unchanged, without overwriting a previous selection."""
    name = Path(name.replace("\\", "/")).name
    if Path(name).suffix.lower() not in UPLOAD_EXTENSIONS:
        raise ValueError("Usar JPG, PNG, TIFF/GeoTIFF, BMP, WebP, GIF, ZIP o 7z.")
    if not len(content):
        raise ValueError("El archivo cargado esta vacio.")
    safe_name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', "_", name)
    directory = file_path(folder)/uuid.uuid4().hex[:12]
    directory.mkdir(parents=True, exist_ok=False)
    path = directory/("imagen_" + safe_name)
    with path.open("wb") as stream:
        stream.write(content)
    return path


def archive_target(root: Path, name: str) -> Path:
    """Resolve one archive member inside its fresh extraction directory."""
    normalized = name.replace("\\", "/")
    parts = PurePosixPath(normalized).parts
    if not parts or normalized.startswith("/") or any(
        part in {"..", "."} or part.endswith((" ", ".")) or
        re.search(r'[<>:"|?*\x00-\x1f]', part) or
        re.fullmatch(r"(?i)(CON|PRN|AUX|NUL|COM[1-9]|LPT[1-9])(?:\..*)?", part)
        for part in parts
    ):
        raise ValueError(f"Ruta no permitida dentro del comprimido: {name!r}")
    destination = root.joinpath(*parts).resolve()
    if not destination.is_relative_to(root.resolve()):
        raise ValueError("El archivo comprimido intenta escribir fuera de su carpeta.")
    return destination


def archive_image_or_sidecar(name: str) -> bool:
    """Keep raster sidecars alongside images so nodata/georeferencing are retained."""
    path = Path(name)
    if "__MACOSX" in path.parts or path.name.startswith("._"):
        return False
    return (path.suffix.lower() in IMAGE_EXTENSIONS | {".msk", ".tfw", ".prj", ".wld", ".jgw", ".pgw"}
            or name.lower().endswith(".aux.xml"))


def extract_image_archive(path: str | Path, max_bytes: int = MAX_ARCHIVE_BYTES) -> list[Path]:
    """Extract regular image files and sidecars, rejecting links and unsafe paths.

    Password protected, multipart and nested archives are not expanded. Contents
    remain isolated per upload and images are returned in deterministic order.
    """
    path = file_path(path)
    destination = path.parent/("extraido_"+uuid.uuid4().hex[:8])
    destination.mkdir()

    def validate_entries(entries: list[tuple[str, int, bool, bool]]) -> list[tuple[str, Path, int]]:
        if len(entries) > MAX_ARCHIVE_MEMBERS:
            raise ValueError("El comprimido tiene demasiados archivos (limite 10000).")
        seen = set()
        selected = []
        for name, size, directory, link in entries:
            target = archive_target(destination, name)
            key = str(target).casefold()
            if key in seen:
                raise ValueError(f"Nombre duplicado dentro del comprimido: {name}")
            seen.add(key)
            if link:
                raise ValueError("El comprimido contiene enlaces; se requieren archivos regulares.")
            if not directory and archive_image_or_sidecar(name):
                selected.append((name, target, size))
        total = sum(size for _, _, size in selected)
        if total > max_bytes:
            raise ValueError(f"El contenido supera el limite de extraccion de {max_bytes/1024**3:g} GiB.")
        if total > shutil.disk_usage(destination).free:
            raise OSError("No hay espacio suficiente para extraer las imagenes.")
        if not any(target.suffix.lower() in IMAGE_EXTENSIONS for _, target, _ in selected):
            raise ValueError("El comprimido no contiene imagenes compatibles.")
        for _, target, _ in selected:
            target.parent.mkdir(parents=True, exist_ok=True)
        return selected

    if path.suffix.lower() == ".zip":
        with zipfile.ZipFile(path) as archive:
            entries = []
            for item in archive.infolist():
                if item.flag_bits & 1:
                    raise ValueError("El ZIP tiene contraseña; cargar una copia sin contraseña.")
                kind = stat.S_IFMT(item.external_attr >> 16)
                entries.append((item.filename, item.file_size, item.is_dir(),
                                kind not in {0, stat.S_IFREG, stat.S_IFDIR}))
            selected = validate_entries(entries)
            total_written = 0
            for name, target, expected in selected:
                written = 0
                with archive.open(name) as src, target.open("xb") as dst:
                    while block := src.read(1024*1024):
                        written += len(block)
                        total_written += len(block)
                        if written > expected or total_written > max_bytes:
                            raise ValueError("El ZIP excedio el tamaño de extraccion declarado.")
                        dst.write(block)
    elif path.suffix.lower() == ".7z":
        try:
            import py7zr
        except ImportError as exc:
            raise RuntimeError("Falta py7zr: ejecutar otra vez la celda de instalacion de este notebook.") from exc
        with py7zr.SevenZipFile(path, mode="r") as archive:
            if archive.needs_password():
                raise ValueError("El 7z tiene contraseña; cargar una copia sin contraseña.")
            links = {entry.filename for entry in archive.files if entry.is_symlink or entry.is_junction or entry.is_socket}
            selected = validate_entries([(item.filename, item.uncompressed, item.is_directory, item.filename in links)
                                         for item in archive.list()])
            archive.extract(path=destination, targets=[name for name, _, _ in selected], recursive=False)
    else:
        raise ValueError("Se esperaba un archivo ZIP o 7z.")
    for _, target, expected in selected:
        if not target.is_file() or target.is_symlink() or target.stat().st_size != expected:
            raise ValueError("La extraccion no coincide con el contenido declarado.")
    return sorted([target for _, target, _ in selected if target.suffix.lower() in IMAGE_EXTENSIONS], key=lambda p: str(p).casefold())


def prepare_upload(name: str, content: bytes | memoryview, folder: str | Path) -> list[Path]:
    """Save a selected image or unpack the images within one ZIP/7z."""
    path = save_uploaded_image(name, content, folder)
    return extract_image_archive(path) if path.suffix.lower() in {".zip", ".7z"} else [path]


class PhotoSource:
    """Decoded photo with EXIF orientation applied; no inferred map coordinates."""
    def __init__(self, path: Path) -> None:
        with Image.open(path) as image:
            self.frames = getattr(image, "n_frames", 1)
            image.seek(0)
            self.rgba = np.array(ImageOps.exif_transpose(image).convert("RGBA"))
        self.height, self.width = self.rgba.shape[:2]
        self.count = 3
        self.crs = None


@contextmanager
def open_source(path: str | Path) -> Iterator[rasterio.io.DatasetReader | PhotoSource]:
    """Windowed TIFFs and common photo formats share the same search pipeline."""
    path = file_path(path)
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        raise ValueError("Formato no admitido. Usar JPG, PNG, TIFF/GeoTIFF, BMP, WebP o GIF.")
    if path.suffix.lower() in {".tif", ".tiff"}:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", rasterio.errors.NotGeoreferencedWarning)
            with rasterio.open(path) as src:
                yield src
    else:
        yield PhotoSource(path)


@dataclass(frozen=True)
class AgroConfig:
    """Explicit RGB radiometry and tiling contract; sizes are native pixels."""
    labels: tuple[str, ...] = ("Flor", "Planta de maleza")
    prompts: tuple[str, ...] = ("flower", "weed plant")
    model: str = "yolov8s-worldv2.pt"
    backend: str = "world"  # 'trained': local detection weights and their exact class names
    tile_size: int = 640
    overlap: int = 192
    batch_size: int = 4
    confidence: float = 0.20
    nms_iou: float = 0.40
    max_det: int = 1000
    bands: tuple[int, int, int] = (1, 2, 3)  # R, G, B (1-based)
    value_range: tuple[float, float] | None = None
    min_size_px: float = 3.0
    min_valid_fraction: float = 0.8
    device: str | None = None
    review_count: int = 40
    max_candidates: int = 250_000
    roi: tuple[int, int, int, int] | None = None  # x, y, width, height

    def validate(self) -> None:
        """Reject ambiguous radiometry, empty classes and invalid tile settings."""
        if not self.labels or len(self.labels) != len(self.prompts):
            raise ValueError("Cada etiqueta necesita un prompt o clase del modelo.")
        if any(not s.strip() for s in (*self.labels, *self.prompts)):
            raise ValueError("Las etiquetas y prompts no pueden estar vacios.")
        if len(set(self.labels)) != len(self.labels) or len(set(self.prompts)) != len(self.prompts):
            raise ValueError("No repetir etiquetas o prompts; evita conteos ambiguos.")
        if self.backend not in {"world", "trained"}:
            raise ValueError("backend debe ser world o trained.")
        if self.tile_size < 32 or self.tile_size % 32 or not 0 <= self.overlap < self.tile_size:
            raise ValueError("Sector multiplo de 32; 0 <= solape < sector.")
        if self.batch_size < 1 or self.max_det < 1 or self.max_candidates < 1 or self.review_count < 0:
            raise ValueError("Lote y limites deben ser positivos; revisiones >= 0.")
        if not 0 < self.confidence <= 1 or not 0 <= self.nms_iou <= 1:
            raise ValueError("Confianza (0,1] e IoU [0,1].")
        if not 0 <= self.min_valid_fraction <= 1 or not math.isfinite(self.min_size_px) or self.min_size_px <= 0:
            raise ValueError("Fraccion valida [0,1]; tamano minimo positivo.")
        if len(self.bands) != 3 or len(set(self.bands)) != 3 or min(self.bands) < 1:
            raise ValueError("Indicar tres bandas distintas R,G,B desde 1.")
        if self.value_range is not None:
            lo, hi = self.value_range
            if not np.isfinite([lo, hi]).all() or hi <= lo:
                raise ValueError("Rango radiometrico invalido.")


@dataclass(frozen=True)
class Hit:
    """Half-open box coordinates on the original orthomosaic."""
    label: str
    confidence: float
    x1: float
    y1: float
    x2: float
    y2: float
    tile_id: int


def starts(length: int, size: int, overlap: int) -> list[int]:
    """Cover all pixels, including a full final tile, with no duplicate origins."""
    if length < 1 or size < 1 or not 0 <= overlap < size:
        raise ValueError("Dimensiones o solape invalidos.")
    return sorted(set(range(0, max(1, length-size+1), size-overlap)) | {max(0, length-size)})


def windows(width: int, height: int, cfg: AgroConfig) -> Iterator[Window]:
    """Generate windows constrained to the configured pixel ROI."""
    x, y, w, h = cfg.roi or (0, 0, width, height)
    if any(not isinstance(v, int) for v in (x, y, w, h)) or min(x, y) < 0 or min(w, h) < 1 or x+w > width or y+h > height:
        raise ValueError("ROI fuera del raster; usar x,y,ancho,alto en pixeles enteros.")
    for row in starts(h, cfg.tile_size, cfg.overlap):
        for col in starts(w, cfg.tile_size, cfg.overlap):
            yield Window(x+col, y+row, min(cfg.tile_size, w-col), min(cfg.tile_size, h-row))


def read_rgb(src: rasterio.io.DatasetReader | PhotoSource, win: Window, cfg: AgroConfig,
             out_shape: tuple[int, int] | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Read one native window, or a bounded preview; preserve alpha/nodata masks."""
    if isinstance(src, PhotoSource):
        x, y, w, h = map(int, (win.col_off, win.row_off, win.width, win.height))
        rgba = src.rgba[y:y+h, x:x+w]
        if max(cfg.bands) > 3:
            raise ValueError("La foto tiene tres canales RGB.")
        if out_shape:
            rgba = cv2.resize(rgba, (out_shape[1], out_shape[0]), interpolation=cv2.INTER_NEAREST)
        valid = rgba[:, :, 3] > 0
        rgb = rgba[:, :, np.array(cfg.bands)-1].copy()
        rgb[~valid] = 0
        # Pillow has already decoded the standard photo to uint8 RGB.
        return np.ascontiguousarray(rgb[:, :, ::-1]), valid
    bands = (1, 1, 1) if src.count <= 2 and tuple(cfg.bands) == (1, 2, 3) else cfg.bands
    if max(bands) > src.count:
        raise ValueError("Las bandas RGB solicitadas no existen en el raster.")
    kwargs = {"out_shape": (3, *out_shape), "resampling": Resampling.nearest} if out_shape else {}
    data = src.read(bands, window=win, masked=True, **kwargs)
    valid = (~np.ma.getmaskarray(data)).all(axis=0) & np.isfinite(data.data).all(axis=0)
    valid &= src.dataset_mask(window=win, **({"out_shape": out_shape} if out_shape else {})) > 0
    if cfg.value_range is None:
        if data.dtype != np.uint8:
            raise ValueError("RGB no uint8: configurar value_range fijo (ej. 0,65535 o 0,1).")
        rgb = data.filled(0).transpose(1, 2, 0).copy()
    else:
        lo, hi = cfg.value_range
        scaled = (data.filled(0).astype(np.float32)-lo) * (255.0/(hi-lo))
        rgb = np.nan_to_num(scaled, nan=0, posinf=255, neginf=0).clip(0, 255).astype(np.uint8).transpose(1, 2, 0)
    rgb[~valid] = 0
    return np.ascontiguousarray(rgb[:, :, ::-1]), valid


class Detector:
    """One model and one vocabulary per run, with explicit trained-class mapping."""
    def __init__(self, cfg: AgroConfig) -> None:
        from ultralytics import YOLO, YOLOWorld
        self.cfg = cfg
        if cfg.backend == "world":
            self.model = YOLOWorld(cfg.model)
            self.model.set_classes(list(cfg.prompts))
            self.class_map = dict(enumerate(cfg.labels))
        else:
            if not Path(cfg.model).is_file():
                raise FileNotFoundError("Modelo entrenado: indicar un archivo local .pt.")
            self.model = YOLO(cfg.model)
            if self.model.task != "detect":
                raise ValueError("Se requieren pesos de deteccion de cajas.")
            names = self.model.names
            inverse = {name: idx for idx, name in names.items()}
            missing = set(cfg.prompts)-set(inverse)
            if missing:
                raise ValueError(f"Clases ausentes en los pesos: {sorted(missing)}; disponibles: {names}")
            self.class_map = {inverse[p]: label for p, label in zip(cfg.prompts, cfg.labels)}

    def __call__(self, images: list[np.ndarray]) -> list[list[tuple[str, float, float, float, float, float]]]:
        """Return label, score and local xyxy. Retry CUDA OOM only."""
        import torch
        try:
            with torch.inference_mode():
                results = list(self.model.predict(
                    source=images, imgsz=self.cfg.tile_size, batch=len(images),
                    conf=self.cfg.confidence, iou=self.cfg.nms_iou,
                    max_det=self.cfg.max_det, device=self.cfg.device,
                    classes=list(self.class_map), verbose=False,
                ))
        except torch.cuda.OutOfMemoryError:
            if len(images) == 1:
                raise
        else:
            output = []
            for result in results:
                rows = result.boxes.data.detach().cpu().numpy()
                if len(rows) >= self.cfg.max_det:
                    raise RuntimeError("Se alcanzo max_det por sector; reducir sector o aumentar max_det para evitar subconteo.")
                output.append([(self.class_map[int(r[5])], float(r[4]), *map(float, r[:4])) for r in rows])
            return output
        torch.cuda.empty_cache()
        mid = len(images)//2
        LOGGER.warning("Memoria GPU insuficiente; dividiendo lote de %s", len(images))
        return self(images[:mid]) + self(images[mid:])


def deduplicate(hits: list[Hit], threshold: float, cell_size: int = 640) -> list[Hit]:
    """Exact greedy class-wise IoU NMS with a spatial grid of kept boxes.

    Avoids all-pairs comparisons for sparse scenes; worst case remains quadratic
    for extremely dense overlapping boxes. Adjacent distinct plants are retained.
    """
    grid: dict[tuple[str, int, int], list[int]] = defaultdict(list)
    kept: list[Hit] = []
    for hit in sorted(hits, key=lambda d: -d.confidence):
        cells = [(hit.label, x, y)
                 for x in range(int(hit.x1//cell_size), int(hit.x2//cell_size)+1)
                 for y in range(int(hit.y1//cell_size), int(hit.y2//cell_size)+1)]
        candidate_ids = {idx for key in cells for idx in grid.get(key, [])}
        area = (hit.x2-hit.x1)*(hit.y2-hit.y1)
        duplicate = False
        for idx in candidate_ids:
            other = kept[idx]
            inter = max(0, min(hit.x2, other.x2)-max(hit.x1, other.x1)) * max(0, min(hit.y2, other.y2)-max(hit.y1, other.y1))
            union = area + (other.x2-other.x1)*(other.y2-other.y1)-inter
            if union > 0 and inter/union > threshold:
                duplicate = True
                break
        if not duplicate:
            for key in cells:
                grid[key].append(len(kept))
            kept.append(hit)
    return sorted(kept, key=lambda d: (d.label, d.y1, d.x1))


def raster_metadata(src: rasterio.io.DatasetReader | PhotoSource) -> dict:
    """Use pixels for ordinary images; retain actual CRS only when declared."""
    if isinstance(src, PhotoSource):
        return {"georeferenced": False, "coordinate_space": "pixeles de imagen orientada segun EXIF",
                "width": src.width, "height": src.height, "crs": None, "crs_wkt": None,
                "transform": None, "gsd_m": None, "dtypes": ["uint8"]*3, "frames": src.frames,
                "frame_processed": 0}
    if src.crs is None:
        return {"georeferenced": False, "coordinate_space": "pixeles del TIFF",
                "width": src.width, "height": src.height, "crs": None, "crs_wkt": None,
                "transform": None, "gsd_m": None, "dtypes": src.dtypes,
                "nodata": str(src.nodata), "frame_processed": 0}
    t = src.transform
    if abs(t.a*t.e-t.b*t.d) < 1e-20 or t.is_identity:
        raise ValueError("Geotransformacion ausente o degenerada.")
    crs = CRS.from_user_input(src.crs)
    gsd = None
    if crs.is_projected and len(crs.axis_info) >= 2:
        fx, fy = (a.unit_conversion_factor for a in crs.axis_info[:2])
        gsd = [math.hypot(t.a*fx, t.d*fy), math.hypot(t.b*fx, t.e*fy)]
    return {"georeferenced": True, "coordinate_space": "pixeles originales y CRS del raster",
            "crs_wkt": crs.to_wkt(), "crs": src.crs.to_string(), "transform": list(t)[:6],
            "width": src.width, "height": src.height, "gsd_m": gsd,
            "dtypes": src.dtypes, "nodata": str(src.nodata), "block_shapes": src.block_shapes}


def write_json(path: Path, data: dict) -> None:
    """Write finite JSON with explicit UTF-8 encoding."""
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")


def export_hits(hits: list[Hit], src: rasterio.io.DatasetReader | PhotoSource, output: Path) -> dict[str, Path]:
    """Always export pixel detections; create GIS layers only with valid CRS."""
    transformer = Transformer.from_crs(src.crs, "EPSG:4326", always_xy=True) if src.crs else None
    points, boxes, rows = [], [], []
    for i, hit in enumerate(hits, 1):
        cx, cy = (hit.x1+hit.x2)/2, (hit.y1+hit.y2)/2
        x, y, lon, lat = None, None, None, None
        if transformer:
            x, y = src.transform * (cx, cy)
            lon, lat = transformer.transform(x, y, errcheck=True)
        row = {"id": i, **asdict(hit), "center_x_px": cx, "center_y_px": cy,
               "x": x, "y": y, "crs": src.crs.to_string() if src.crs else None,
               "lon": lon, "lat": lat, "revision": "pendiente"}
        rows.append(row)
        if transformer is None:
            continue
        points.append({"type": "Feature", "properties": row, "geometry": {"type": "Point", "coordinates": [lon, lat]}})
        ring = []
        for col, r in [(hit.x1, hit.y1), (hit.x2, hit.y1), (hit.x2, hit.y2), (hit.x1, hit.y2), (hit.x1, hit.y1)]:
            ring.append(list(transformer.transform(*(src.transform*(col, r)), errcheck=True)))
        boxes.append({"type": "Feature", "properties": row, "geometry": {"type": "Polygon", "coordinates": [ring]}})
    artifacts = {"csv": output/"detecciones.csv", "json": output/"detecciones.json"}
    fields = ["id", *Hit.__dataclass_fields__, "center_x_px", "center_y_px", "x", "y", "crs", "lon", "lat", "revision"]
    with artifacts["csv"].open("w", encoding="utf-8-sig", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    write_json(artifacts["json"], {"image": raster_metadata(src), "detections": rows})
    if transformer:
        for name, features in (("puntos", points), ("cajas", boxes)):
            artifacts[name] = output/f"{name}.geojson"
            write_json(artifacts[name], {"type": "FeatureCollection", "features": features})
    return artifacts


def annotated_preview(src: rasterio.io.DatasetReader | PhotoSource, cfg: AgroConfig,
                      hits: list[Hit], path: Path, max_side: int = 2048) -> dict:
    """Save a bounded marked preview. Inference always uses native image pixels."""
    scale = min(1.0, max_side/max(src.width, src.height))
    w, h = max(1, round(src.width*scale)), max(1, round(src.height*scale))
    image, _ = read_rgb(src, Window(0, 0, src.width, src.height), cfg, out_shape=(h, w))
    sx, sy = w/src.width, h/src.height
    for idx, hit in enumerate(hits, 1):
        color = (0, 195, 255) if hit.label == "Flor" else (50, 220, 40)
        x1, y1 = int(hit.x1*sx), int(hit.y1*sy)
        x2, y2 = max(x1, math.ceil(hit.x2*sx)-1), max(y1, math.ceil(hit.y2*sy)-1)
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 1)
        cv2.putText(image, f"{idx}", (x1, max(10, y1-3)), cv2.FONT_HERSHEY_SIMPLEX, .4, color, 1, cv2.LINE_AA)
    ok, encoded = cv2.imencode(".jpg", image, [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    if not ok:
        raise OSError("No se pudo guardar la imagen marcada.")
    encoded.tofile(str(path))
    return {"width": w, "height": h, "scale_x": sx, "scale_y": sy,
            "note": "Vista previa sin georreferenciacion; IDs corresponden al CSV. No usada para inferencia."}


def run_agro(source: str | Path, output_root: str | Path, cfg: AgroConfig | None = None,
             predictor: Callable | None = None) -> dict[str, Path]:
    """Process windows sequentially; save raw candidates and a completion manifest.

    A supplied predictor is for integration tests or alternate compatible backends.
    TIFF pixel memory is bounded by the batch and preview; photos are decoded in RAM.
    """
    cfg = cfg or AgroConfig()
    cfg.validate()
    source = file_path(source).resolve(strict=True)
    start_time = time.perf_counter()
    with open_source(source) as src:
        metadata = raster_metadata(src)
        # Validate ROI/radiometry before loading neural weights.
        first = next(windows(src.width, src.height, cfg))
        read_rgb(src, first, cfg)
        total = sum(1 for _ in windows(src.width, src.height, cfg))
        output = file_path(output_root)/datetime.now().strftime("agro_%Y%m%d_%H%M%S_%f")
        output.mkdir(parents=True, exist_ok=False)
        manifest_path = output/"ejecucion.json"
        versions = {}
        for name in ("numpy", "rasterio", "pyproj", "ultralytics", "torch"):
            try:
                versions[name] = importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                versions[name] = "no disponible"
        manifest = {"status": "running", "source": str(source), "source_bytes": source.stat().st_size,
                    "source_mtime_ns": source.stat().st_mtime_ns, "config": asdict(cfg), "raster": metadata,
                    "versions": versions, "started_utc": datetime.now(timezone.utc).isoformat(),
                    "predictor": "injected" if predictor is not None else cfg.backend,
                    "scope": "ROI" if cfg.roi else "raster completo", "tiles_total": total,
                    "limits": "Detecciones exploratorias; sin validacion agronomica. Cajas no son contornos de plantas. No calcular densidad desde el area rectangular con nodata."}
        write_json(manifest_path, manifest)
        raw_path = output/"candidatos.jsonl"
        candidates: list[Hit] = []
        skipped = 0
        processed = 0
        try:
            detect = predictor if predictor is not None else Detector(cfg)
            with raw_path.open("w", encoding="utf-8") as raw:
                batch = []

                def flush() -> None:
                    """Infer and transform exactly one pending batch into mosaic pixels."""
                    nonlocal processed
                    results = detect([item[2] for item in batch])
                    if len(results) != len(batch):
                        raise RuntimeError("El detector no devolvio un resultado por sector.")
                    for (tile_id, win, bgr, valid), rows in zip(batch, results):
                        h, w = valid.shape
                        for label, score, x1, y1, x2, y2 in rows:
                            if label not in cfg.labels:
                                raise ValueError(f"Clase inesperada del detector: {label}")
                            if not np.isfinite([score, x1, y1, x2, y2]).all() or not cfg.confidence <= score <= 1:
                                continue
                            x1, x2 = np.clip([x1, x2], 0, w)
                            y1, y2 = np.clip([y1, y2], 0, h)
                            if min(x2-x1, y2-y1) < cfg.min_size_px:
                                continue
                            region = valid[int(y1):math.ceil(y2), int(x1):math.ceil(x2)]
                            if not valid[min(h-1, int((y1+y2)/2)), min(w-1, int((x1+x2)/2))] or region.mean() < cfg.min_valid_fraction:
                                continue
                            hit = Hit(label, float(score), float(x1+win.col_off), float(y1+win.row_off), float(x2+win.col_off), float(y2+win.row_off), tile_id)
                            candidates.append(hit)
                            raw.write(json.dumps(asdict(hit), ensure_ascii=False, allow_nan=False)+"\n")
                            if len(candidates) > cfg.max_candidates:
                                raise RuntimeError("Limite de candidatos alcanzado; procesar por ROI o ajustar confianza.")
                        processed += 1
                    raw.flush()
                    batch.clear()
                    LOGGER.info("Sectores %s/%s; omitidos %s; candidatos %s", processed+skipped, total, skipped, len(candidates))

                for tile_id, win in enumerate(windows(src.width, src.height, cfg)):
                    bgr, valid = read_rgb(src, win, cfg)
                    if not valid.any():
                        skipped += 1
                        continue
                    batch.append((tile_id, win, bgr, valid))
                    if len(batch) == cfg.batch_size:
                        flush()
                if batch:
                    flush()
            hits = deduplicate(candidates, cfg.nms_iou, cfg.tile_size)
            artifacts = export_hits(hits, src, output)
            artifacts["imagen"] = output/"imagen_marcada.jpg"
            preview = annotated_preview(src, cfg, hits, artifacts["imagen"])
            review = output/"recortes_revision"
            review.mkdir()
            # Evenly distributed sample in spatially sorted hits; not an accuracy sample.
            ids = np.linspace(0, len(hits)-1, min(cfg.review_count, len(hits)), dtype=int) if hits else []
            for idx in ids:
                hit = hits[int(idx)]
                x, y = max(0, int(hit.x1)-30), max(0, int(hit.y1)-30)
                w, h = min(src.width-x, math.ceil(hit.x2)-x+30), min(src.height-y, math.ceil(hit.y2)-y+30)
                bgr, _ = read_rgb(src, Window(x, y, w, h), cfg)
                cv2.rectangle(bgr, (int(hit.x1)-x, int(hit.y1)-y), (math.ceil(hit.x2)-x-1, math.ceil(hit.y2)-y-1), (0, 0, 255), 1)
                ok, data = cv2.imencode(".jpg", bgr)
                if not ok:
                    raise OSError("No se pudo guardar recorte de revision.")
                data.tofile(str(review/f"{int(idx)+1:06d}.jpg"))
            manifest.update(status="completed", tiles_processed=processed, tiles_skipped_nodata=skipped,
                            candidates=len(candidates), detections=len(hits),
                            counts={label: Counter(h.label for h in hits)[label] for label in cfg.labels}, preview=preview,
                            elapsed_seconds=round(time.perf_counter()-start_time, 3))
            write_json(manifest_path, manifest)
            artifacts.update(manifest=manifest_path, candidatos=raw_path)
            zip_path = output/"reporte_agro.zip"
            with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
                for path in [*artifacts.values(), *sorted(review.glob("*.jpg"))]:
                    archive.write(path, path.relative_to(output))
            artifacts["zip"] = zip_path
            return artifacts
        except Exception as exc:
            manifest.update(status="failed", error=f"{type(exc).__name__}: {exc}", tiles_processed=processed,
                            tiles_skipped_nodata=skipped, candidates=len(candidates))
            write_json(manifest_path, manifest)
            raise


def main() -> None:
    """Run from PowerShell, optionally reading all settings from JSON."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("input", type=Path)
    parser.add_argument("--output", type=Path, default=Path("resultados_agro"))
    parser.add_argument("--config", type=Path)
    args = parser.parse_args()
    cfg = AgroConfig(**json.loads(args.config.read_text(encoding="utf-8"))) if args.config else AgroConfig()
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
    print(json.dumps({k: str(v) for k, v in run_agro(args.input, args.output, cfg).items()}, indent=2, ensure_ascii=False))



## 3. Configuración (lista para usar)

In [ ]:
# Ejecutar tal como está. La imagen se selecciona con el botón Cargar imagen.
# Estas opciones avanzadas son opcionales; no hay que escribir rutas.
CONFIG = AgroConfig(
    labels=("Flor", "Planta de maleza"),
    prompts=("flower", "weed plant"),
    tile_size=640, overlap=192, batch_size=4,
    confidence=0.20, nms_iou=0.40,
    bands=(1, 2, 3), value_range=None,
    roi=None,  # Prueba pequeña: (x, y, ancho, alto), en píxeles del raster.
    # backend="trained", model="mis_pesos.pt",  # prompts = nombres exactos de sus clases.
)


## Interfaz

In [ ]:
"""Direct image upload for Colab/Jupyter; no user-entered paths required."""
from dataclasses import replace
import html
import traceback
import ipywidgets as widgets
from IPython.display import display, clear_output, FileLink, Image as DisplayImage

try:
    from google.colab import files as colab_files
except ImportError:
    colab_files = None

# Internal session folders are managed automatically.
_carpeta_cargas = Path.cwd()/"cargas_agro"
_carpeta_resultados = Path.cwd()/"resultados_agro"
imagen_cargada = None
artefactos_agro = {}
objetivos_agro = widgets.SelectMultiple(
    options=list(zip(CONFIG.labels, CONFIG.prompts)), value=CONFIG.prompts,
    description="Buscar", layout=widgets.Layout(width="95%", height="90px"))
confianza_agro = widgets.FloatSlider(value=CONFIG.confidence, min=.01, max=.9, step=.01, description="Confianza")
rango_agro = widgets.Dropdown(options=[("Imagen habitual (8 bits)", None),
                                      ("TIFF 16 bits: 0 a 65535", (0, 65535)),
                                      ("TIFF float: 0 a 1", (0, 1))],
                             value=CONFIG.value_range, description="Rango")
estado_agro = widgets.HTML("<p>1. Cargá una imagen. 2. Elegí qué buscar. 3. Pulsá Buscar.</p>")
selector_imagen = widgets.Dropdown(options=[], description="Imagen", disabled=True,
                                  layout=widgets.Layout(width="95%"))
procesar_agro = widgets.Button(description="Buscar en la imagen", button_style="success", disabled=True)
descargar_agro = widgets.Button(description="Descargar ZIP", disabled=True)
salida_agro = widgets.Output()


def configuracion_agro() -> AgroConfig:
    """Read controls without requiring code edits for ordinary searches."""
    mapping = dict(zip(CONFIG.prompts, CONFIG.labels))
    selected = tuple(objetivos_agro.value)
    cfg = replace(CONFIG, labels=tuple(mapping[p] for p in selected), prompts=selected,
                  confidence=confianza_agro.value, value_range=rango_agro.value)
    cfg.validate()
    return cfg


def reiniciar_carga(limpiar_lista: bool = True) -> None:
    """Invalidate old results whenever another upload starts or fails."""
    global imagen_cargada, artefactos_agro
    imagen_cargada = None
    artefactos_agro = {}
    procesar_agro.disabled = True
    descargar_agro.disabled = True
    estado_agro.value = "<p>Seleccioná una imagen para empezar.</p>"
    if limpiar_lista:
        selector_imagen.options = []
        selector_imagen.disabled = True


def aceptar_carga(name: str, content: bytes | memoryview) -> None:
    """Accept one image or archive, then offer all extracted images for selection."""
    reiniciar_carga()
    with salida_agro:
        clear_output()
        try:
            paths = prepare_upload(name, content, _carpeta_cargas)
            base = Path(os.path.commonpath([str(p.parent) for p in paths]))
            options = [(str(path.relative_to(base)), str(path)) for path in paths]
            selector_imagen.disabled = False
            selector_imagen.options = options
            selector_imagen.value = options[0][1]
            if len(paths) > 1:
                print(f"Se encontraron {len(paths)} imágenes. Elegí cuál analizar en la lista Imagen.")
        except Exception as exc:
            print(f"No se pudo cargar el archivo: {exc}")
            estado_agro.value = "<p>No se pudo leer el archivo. Cargá una imagen, ZIP o 7z válido.</p>"


def mostrar_imagen_seleccionada(change) -> None:
    """Switch the active image and invalidate any report of the previous selection."""
    global imagen_cargada
    if not change["new"]:
        return
    reiniciar_carga(limpiar_lista=False)
    path = Path(change["new"])
    with salida_agro:
        clear_output()
        try:
            with open_source(path) as src:
                meta = raster_metadata(src)
                text = f"{path.name} · {src.width:,} × {src.height:,} px"
                text += f" · {meta['crs']}" if meta["georeferenced"] else " · resultados en píxeles"
                cfg = replace(CONFIG, value_range=rango_agro.value)
                try:
                    preview_path = path.parent/"vista_previa.jpg"
                    annotated_preview(src, cfg, [], preview_path, max_side=1200)
                except ValueError as exc:
                    print(f"Imagen cargada. Ajustá el selector Rango antes de buscar: {exc}")
                else:
                    display(DisplayImage(data=preview_path.read_bytes(), width=850))
                if meta.get("frames", 1) > 1:
                    print("Imagen animada: se analizará sólo el primer fotograma.")
                if meta["gsd_m"]:
                    print("Resolución del archivo (cm/píxel):", [round(v*100, 3) for v in meta["gsd_m"]])
            imagen_cargada = path
            estado_agro.value = f"<p><b>Imagen lista:</b> {html.escape(text)}</p>"
            procesar_agro.disabled = False
        except Exception as exc:
            print(f"No se pudo leer la imagen seleccionada: {exc}")
            estado_agro.value = "<p>Elegí otra imagen o cargá un archivo válido.</p>"


def subir_colab(_):
    """Open Colab's computer file picker and consume its returned bytes."""
    reiniciar_carga()
    with salida_agro:
        clear_output()
        try:
            uploaded = colab_files.upload()
            if not uploaded:
                print("No se seleccionó ninguna imagen.")
                return
            if len(uploaded) != 1:
                print("Seleccioná un archivo por carga: una imagen, ZIP o 7z.")
                return
            name, content = next(iter(uploaded.items()))
            aceptar_carga(name, content)
        except Exception as exc:
            print(f"No se pudo completar la carga: {exc}")


def subir_jupyter(change):
    """Consume ipywidgets 8 upload records and release the widget content."""
    records = change["new"]
    if not records:
        return
    reiniciar_carga()
    try:
        if len(records) != 1:
            with salida_agro:
                print("Seleccioná un archivo por carga: una imagen, ZIP o 7z.")
            return
        record = records[0]
        aceptar_carga(record["name"], record["content"])
    finally:
        cargar_agro.value = ()


def ejecutar_agro(_):
    global artefactos_agro
    artefactos_agro = {}
    procesar_agro.disabled = True
    descargar_agro.disabled = True
    cargar_agro.disabled = True
    selector_imagen.disabled = True
    with salida_agro:
        clear_output()
        try:
            if imagen_cargada is None:
                print("Primero pulsá Cargar imagen y seleccioná un archivo de tu computadora.")
                return
            cfg = configuracion_agro()
            print("Buscando en la imagen cargada…")
            artefactos_agro = run_agro(imagen_cargada, _carpeta_resultados, cfg)
            resumen = json.loads(artefactos_agro["manifest"].read_text(encoding="utf-8"))
            print("Detecciones para revisar:", resumen["counts"])
            display(DisplayImage(data=artefactos_agro["imagen"].read_bytes(), width=1000))
            print("Los números de la imagen y los recortes corresponden al ID del CSV.")
            if resumen["raster"]["georeferenced"]:
                print("Incluye puntos y cajas GeoJSON para QGIS.")
            else:
                print("Imagen sin georreferenciación: posiciones en píxeles; no se asignan coordenadas GPS.")
            if colab_files is None:
                for name in ("imagen", "csv", "zip"):
                    display(FileLink(str(artefactos_agro[name])))
            descargar_agro.disabled = False
        except Exception:
            traceback.print_exc()
        finally:
            procesar_agro.disabled = imagen_cargada is None
            cargar_agro.disabled = False
            selector_imagen.disabled = not bool(selector_imagen.options)


def bajar_agro(_):
    path = artefactos_agro.get("zip")
    if path is not None:
        if colab_files is not None:
            colab_files.download(str(path))
        else:
            with salida_agro:
                display(FileLink(str(path)))


if colab_files is not None:
    cargar_agro = widgets.Button(description="Cargar imagen / ZIP / 7z", button_style="info", icon="upload",
                                 layout=widgets.Layout(width="240px"))
    cargar_agro.on_click(subir_colab)
else:
    cargar_agro = widgets.FileUpload(accept=",".join(sorted(UPLOAD_EXTENSIONS)), multiple=False,
                                     description="Cargar imagen / ZIP / 7z", button_style="info",
                                     layout=widgets.Layout(width="240px"))
    cargar_agro.observe(subir_jupyter, names="value")
procesar_agro.on_click(ejecutar_agro)
descargar_agro.on_click(bajar_agro)
selector_imagen.observe(mostrar_imagen_seleccionada, names="value")
display(widgets.HTML("<h3>Buscador agro · imágenes y ortomosaicos</h3><p>Cargá JPG, PNG, TIFF/GeoTIFF, BMP, WebP, GIF o un ZIP/7z con imágenes. Si hay varias, elegí una de la lista.</p>"))
display(cargar_agro, selector_imagen, estado_agro, objetivos_agro, confianza_agro, rango_agro)
display(widgets.HBox([procesar_agro, descargar_agro]), salida_agro)


## Uso y validación con tus imágenes

1. Instalar dependencias y ejecutar el motor.
2. Ejecutar configuración e interfaz. Pulsar **Cargar imagen / ZIP / 7z** y seleccionar un archivo de tu computadora. Los comprimidos se extraen automáticamente; si hay varias imágenes, elegir una en la lista **Imagen**.
3. Elegir flores, plantas de maleza o ambas; ajustar confianza si hace falta. Para JPG/PNG habituales, dejar el rango de 8 bits. Para TIFF uint16/float, seleccionar el rango que corresponda a sus valores reales.
4. Pulsar **Buscar en la imagen**. Se muestra una imagen marcada con IDs y el conteo por clase.
5. Pulsar **Descargar ZIP**. Contiene imagen marcada, CSV, JSON y recortes. Sólo los GeoTIFF con CRS válido agregan capas GeoJSON para QGIS.
6. Revisar `recortes_revision`: cada nombre coincide con el ID del CSV. Las cajas no son contornos de plantas. Si se configuró una ROI avanzada, el conteo corresponde sólo a ella.

La carga transfiere el archivo al entorno del notebook, sin Drive. Los archivos grandes pueden tardar y requieren memoria durante la transferencia.
ZIP y 7z: incluye imágenes en subcarpetas y conserva archivos auxiliares del TIFF. Se analiza una imagen elegida por vez.
Se admiten comprimidos completos, sin contraseña; no se abren volúmenes partidos ni comprimidos dentro de otros comprimidos.
La extracción tiene un límite de 20 GiB y 10000 entradas; se informa un error si no hay espacio o se excede el límite.
Los TIFF se procesan por ventanas; las fotos normales se decodifican en memoria. GIF animado y TIFF multipágina: sólo primer fotograma/página.
JPG/PNG respetan la orientación EXIF. Las coordenadas en píxeles y la imagen marcada corresponden a esa orientación.
La imagen marcada se limita a 2048 px de lado para verla; la detección usa sectores a resolución original.
El GPS EXIF de una foto no basta para ubicar cada planta y no se transforma en coordenadas de objetos.

Con 1–2 cm/px, un objeto de 10 cm ocupa aproximadamente 5–10 píxeles;
eso puede ser insuficiente para distinguir su clase. No se infiere un GSD a partir de ese supuesto.
Los recortes mantienen resolución; una imagen previa reducida no se utiliza para detectar.
La eliminación de duplicados usa IoU por clase y puede conservar detecciones parciales muy distintas en bordes.
No se calcula densidad por hectárea sin definir superficie válida y validar conteos.

Si se dispone de ejemplos etiquetados, comparar con una línea base sencilla y evaluar por lotes/vuelos separados
antes de entrenar o elegir pesos especializados. No dividir recortes solapados entre entrenamiento y prueba.
El soporte `trained` permite cargar un detector propio sin tratar un modelo genérico como especialista.

## Estado de verificación

Se probaron carga directa, JPG/PNG/TIFF/BMP/WebP/GIF, extracción real de ZIP y 7z, selección de imágenes, exportación, nodata y coordenadas con datos sintéticos.
Los callbacks de carga Colab/Jupyter se probaron con entradas controladas; no se abrió una sesión interactiva remota de Colab.
La inferencia real no se ejecutó en la PC de preparación: PyTorch presentó un error de DLL y faltaba Ultralytics.
La instalación fijada es una receta de ejecución, todavía no un entorno completo certificado.


Fuentes técnicas consultadas el 2026-09-21: [YOLO-World](https://docs.ultralytics.com/models/yolo-world/), [lectura por ventanas de Rasterio](https://rasterio.readthedocs.io/en/stable/topics/windowed-rw.html).